In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('analysis_c2pt_2')

projs=['P0', 'Px', 'Py', 'Pz']
inserts=['tt', 'tx', 'ty', 'tz', 'xx', 'xy', 'xz', 'yy', 'yz', 'zz']
enss=['b','c','d','e']

In [2]:
ens2c2pt={}; ens2moms_2pt={}; ens2c2pt0={}; ens2Njk={}
for ens in enss:
    basepath=f'/p/project1/ngff/li47/code/projectData/05_moments/{yu.ens2full[ens]}/data_merge/'
    path=f'{basepath}disc_2pt.h5'
    with h5py.File(path) as f:
        moms_2pt=yu.moms2list(f['moms'])
        c2pt=yu.jackknife(np.real(f['data/N_N'][:,:,:]))
        
    ens2moms_2pt[ens]=moms_2pt
    ens2c2pt[ens]=c2pt
    ens2c2pt0[ens]=c2pt[:,:,moms_2pt.index([0,0,0])]
    ens2Njk[ens]=len(c2pt)
    print(ens,len(c2pt))

b 725
c 400
d 493
e 460


In [3]:
ens2tminss={
        'b':[range(8,25+1),range(1,10+1),range(1,5+1)],
        'c':[range(8,29+1),range(1,16+1),range(1,6+1)],
        'd':[range(8,33+1),range(1,18+1),range(1,7+1)],
        'e':[range(8,39+1),range(1,18+1),range(1,7+1)],
    }
ens2tminss_large={
        'b':[range(8,25+1),range(1,9+1),range(1,1+1)],
        'c':[range(8,29+1),range(1,10+1),range(1,1+1)],
        'd':[range(8,33+1),range(1,11+1),range(1,1+1)],
        'e':[range(8,39+1),range(1,13+1),range(1,1+1)],
    } # used for large momenta

mN_ref=(yu.m_proton+yu.m_neutron)/2

path='data_aux/DmN_ChPT.csv'
df=pd.read_csv(path, index_col=0)
def func(ele):
    m,e=ele.split(',')
    m=m.split('(')[-1]
    e=e.split(')')[0]
    return float(m),float(e)

dic_DmN_ChPT={(row_label, col_label): func(df.loc[row_label, col_label])
    for row_label in df.index
    for col_label in df.columns
}

ens2DmN_N2LO={ens:dic_DmN_ChPT[(yu.ens2label[ens],'DmN_N2LO')] for ens in enss}
ens2DmN_N2LO={ens:yu.jackknife_pseudo(ens2DmN_N2LO[ens][0],ens2DmN_N2LO[ens][1],ens2Njk[ens])[:,0] for ens in enss}
print({ens:yu.jackme_un2str(ens2DmN_N2LO[ens]) for ens in enss})

{'b': '-4.50(38)', 'c': '-1.32(47)', 'd': '-5.33(45)', 'e': '-1.24(44)'}


# 0 mom

In [11]:
for ens in enss:
    meff=yu.jackmap(yu.c2pt2meff,ens2c2pt0[ens])
    tminss=ens2tminss[ens]
    fitss_2pt=yu.doFits_meff_nst(meff,tminss,[0.4,0.5,2,0.8,1],downSampling=1,label=f'meff_{ens}',debugQ=True)

In [12]:
ens='b'
fitss=yu.getFits(f'meff_{ens}')
nst2tminphy={}
for nst,fits in zip([1,2,3],fitss):
    # if nst==1:
    #     fits=[fit for fit in fits if fit[0]*yu.ens2a[ens]>1.3]
    pars_jk,probs_jk=yu.jackMA(fits)
    probs=np.mean(probs_jk,axis=0)
    ind=np.argmax(probs)
    tmin=fits[ind][0]
    nst2tminphy[nst]=tmin*yu.ens2a[ens]

nst2ens2tmin={}
for nst in nst2tminphy.keys():
    nst2ens2tmin[nst]={}
    for ens in enss:
        nst2ens2tmin[nst][ens]=round(nst2tminphy[nst]/yu.ens2a[ens])

    print(f'{nst}st fit:',f'{nst2tminphy[nst]:.02f} fm',nst2ens2tmin[nst])

1st fit: 1.27 fm {'b': 16, 'c': 19, 'd': 22, 'e': 26}
2st fit: 0.64 fm {'b': 8, 'c': 9, 'd': 11, 'e': 13}
3st fit: 0.32 fm {'b': 4, 'c': 5, 'd': 6, 'e': 6}


In [13]:
figs=[]; ens2pars_jk_meff1st={}; ens2pars_jk_meff2st={}; ens2pars_jk_meff3st={}
for ens in enss:
    meff=yu.jackmap(yu.c2pt2meff,ens2c2pt0[ens])
    tminss=ens2tminss[ens]
    
    selections={f'{nst}st':nst2ens2tmin[nst][ens] for nst in [1,2,3]}
    
    fitss_2pt=yu.getFits(f'meff_{ens}')
    fig,axd,result=yu.makePlot_2pt_SimoneStyle(meff,fitss_2pt,xunit=yu.ens2a[ens],yunit=yu.ens2aInv[ens]/1000,E0_ref=mN_ref/1000,ylims='std_N',\
        selection=selections)
    fig.suptitle(yu.ens2full[ens])
    yu.finalizePlot(closeQ=True)
    figs.append(fig) 
    
    ens2pars_jk_meff1st[ens]=result['1st']
    ens2pars_jk_meff2st[ens]=result['2st']
    ens2pars_jk_meff3st[ens]=result['3st']
    

ens2dats=[{ens:ens2pars_jk_meffnst[ens][:,0]*yu.ens2aInv[ens] for ens in enss} for ens2pars_jk_meffnst in [ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]]
fitss=[yu.doFits_continuumExtrapolation(ens2dat,lat_a2s_plt=yum.lat_a2s_plt) for ens2dat in ens2dats]
matrix_dic=[{
    'ens2dat':ens2dats[ist],
    'fit:[fits,lat_a2s_plt]':[fitss[ist],yum.lat_a2s_plt]
    } for ist in range(len(ens2dats))]
fig,axs=yu.makePlot_continuumExtrapolation(matrix_dic,shows=['MA'])
yu.addColHeader(axs,['1st','2st','3st'])
ax=axs[0,0]
ax.set_ylim([920,960])
ax.set_ylabel(r'$m_N$ [MeV]')
for icol in range(len(matrix_dic)):
    ax=axs[0,icol]
    yu.addRefLine(ax,mN_ref,label=r'$m_N^{\mathrm{exp}}=$'+'%0.3f'%mN_ref)
    ax.legend(fontsize=16)  
yu.finalizePlot(closeQ=True)
figs.append(fig)

ens2dats=[{ens:ens2pars_jk_meffnst[ens][:,0]*yu.ens2aInv[ens]+ens2DmN_N2LO[ens] for ens in enss} for ens2pars_jk_meffnst in [ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]]
fitss=[yu.doFits_continuumExtrapolation(ens2dat,lat_a2s_plt=yum.lat_a2s_plt) for ens2dat in ens2dats]
matrix_dic=[{
    'ens2dat':ens2dats[ist],
    'fit:[fits,lat_a2s_plt]':[fitss[ist],yum.lat_a2s_plt]
    } for ist in range(len(ens2dats))]
fig,axs=yu.makePlot_continuumExtrapolation(matrix_dic,shows=['MA'])
yu.addColHeader(axs,['1st','2st','3st'])
ax=axs[0,0]
ax.set_ylim([920,960])
ax.set_ylabel(r'$m_N$ [MeV]')
for icol in range(len(matrix_dic)):
    ax=axs[0,icol]
    yu.addRefLine(ax,mN_ref,label=r'$m_N^{\mathrm{exp}}=$'+'%0.3f'%mN_ref)
    ax.legend(fontsize=16)  
fig.suptitle(r'corrected using $O(p^3)$ ChPT')
yu.finalizePlot(closeQ=True)
figs.append(fig)

yu.makePDF('meff',figs)